[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaskarjitsarmah/RL-Agents-Workshop-LLM/blob/main/notebooks/NB2_data_and_star_warm_start.ipynb)

# NB2 - Data: A Verifiable Task Generator, and a STaR Warm Start

Two problems, one notebook.

**Problem 1: 24 training tasks.** GRPO updates 18M LoRA parameters from sampled
reward variance. Two dozen prompts is not a training set.

**Problem 2: the reward is only as good as the gold.** A single wrong gold does
not merely lose one example - it *inverts* the gradient on that prompt,
punishing the policy for being right. Asking a model to write NL/SQL pairs would
make gold quality the weakest link in the entire repo.

So we never generate gold. We **construct** it:

> a hand-verified SQL skeleton + slot values drawn from the live database
> -> the *same* values formatted into both the question and the query

`SELECT name FROM customers WHERE city='{city}'` with `city='Pune'` is correct by
construction, for every value of `city`. The only thing a human must check is the
skeleton - and there are 49 of them, once.

Then we let the model bootstrap from its own successes (**STaR**): sample k
completions, keep the ones the verifiable reward marks correct, fine-tune on
those. No labels, no teacher. *The reward is the filter.*

> **Restart the runtime before this notebook.** Colab does not free GPU memory between notebooks, and a leftover model from the previous one is the most common cause of an out-of-memory error halfway through a training run.
>
> *Runtime -> Restart session*, then run the setup cell below.

In [ ]:
# --- Setup. Safe to re-run. ---------------------------------------------
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/RL-Agents-Workshop-LLM"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/bhaskarjitsarmah/RL-Agents-Workshop-LLM.git", "/content/RL-Agents-Workshop-LLM"], check=True)
    os.chdir("/content/RL-Agents-Workshop-LLM")
    # Colab's own keyring, read BEFORE preflight computes CAP -- otherwise the
    # notebook decides "no W&B key" while the key sits unread in the sidebar.
    # Absent secrets are normal, not an error: everything downgrades gracefully.
    try:
        from google.colab import userdata
        for _k in ("WANDB_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"):
            try:
                os.environ.setdefault(_k, userdata.get(_k) or "")
            except Exception:
                pass
    except Exception:
        pass
    for _k in [k for k, v in list(os.environ.items()) if v == ""]:
        del os.environ[_k]          # empty != set; CAP tests truthiness

    # Install with uv, not pip: same resolution, several times faster on Colab.
    # The CORE layers on top of Colab's torch and never replaces it -- see the
    # header of requirements-colab.txt for why pinning torch broke this before.
    print("Installing the training stack with uv (1-2 min the first time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    def _uv(*pkgs):
        return subprocess.run([sys.executable, "-m", "uv", "pip", "install",
                               "--system", "-q", *pkgs]).returncode

    # CORE -- must succeed. Fail loudly instead of continuing on stock packages:
    # a swallowed install failure surfaces 10 cells later as a dtype or
    # bitsandbytes error that names nothing resembling its cause.
    if _uv("-r", "requirements-colab.txt") != 0:
        print("*** CORE INSTALL FAILED -- scroll up for the uv error. ***")
        raise SystemExit("core install failed -- see WORKSHOP_GUIDE.md")

    # NB5's openpipe-art, best-effort: it can fail to resolve, and NB5 falls back
    # to a pre-baked run if it is absent. A failure here must not break the core.
    _uv("openpipe-art>=0.4.0")

    # Unsloth: ~2x faster LoRA on a T4, which is the difference between NB3
    # fitting in a lunch break and not. Installed HERE rather than in
    # requirements-colab.txt, and UNPINNED.
    #   * unpinned, because the old `unsloth==2024.12.4` pin required torch
    #     2.5.1 and was what made the entire install abort;
    #   * here rather than in the requirements file, because this is the one
    #     dependency heavy enough to fail on the day, and a failure has to
    #     degrade to the transformers + bitsandbytes path, not kill the core.
    # Skip it with:  os.environ["USE_UNSLOTH"] = "0"  above this cell.
    if os.environ.get("USE_UNSLOTH") != "0":
        if _uv("unsloth", "unsloth_zoo") != 0:
            print("unsloth did not install -- continuing on transformers + "
                  "bitsandbytes. Same adapter, slower. This is not an error.")

    # Unsloth CAN drag a different torch in. If it did, the kernel must restart
    # before anything imports torch, or you get a cryptic CUDA error later.
    from importlib.metadata import version as _ver
    if "torch" in sys.modules and sys.modules["torch"].__version__ != _ver("torch"):
        print("=" * 68)
        print("  torch was replaced. Runtime -> Restart session, then Run all again.")
        print("=" * 68)
        raise SystemExit("restart required -- see the message above")
else:
    # Run from the REPO ROOT in both environments, so every relative path in
    # every notebook ("data/...") means the same thing whether you are on Colab
    # (cwd = repo root) or opened the file locally from notebooks/.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

# Results that outlive the VM. Every Colab notebook is a SEPARATE runtime, so
# NB3 trains the GRPO curve into its own /content and NB5 -- a different VM --
# cannot see it. Anything one notebook computes for another has to land
# somewhere shared, and Drive is the only such place on free Colab.
# Set RESULTS_DIR before importing llm_utils: it is read at import time.
if IN_COLAB and os.environ.get("USE_DRIVE", "1") != "0":
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        _rd = "/content/drive/MyDrive/rl-workshop-results"
        os.makedirs(_rd, exist_ok=True)
        os.environ["RESULTS_DIR"] = _rd
        print(f"Results -> {_rd} (shared across notebooks, survives restarts)")
    except Exception as _e:
        print(f"Drive not mounted ({_e}). Results stay in this VM only, so a")
        print("later notebook will not see what this one computes. Not fatal.")

from llm_utils import (build_db, preflight, capability, load_result,
                       report_number, save_result)
from llm_utils.plotting import use_house_style
import matplotlib.pyplot as plt

CAP = preflight()
use_house_style()
print("Database ready at:", build_db())
if not CAP["gpu"]:
    print()
    print("No GPU detected -> REPLAY MODE.")
    print("Training cells will load pre-baked runs; every chart still renders.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
def baked(key, how):
    """Load a pre-baked run, or explain exactly how to produce it.

    Returns None when the artifact is missing. Callers must check -- we would
    rather show no chart than an invented one.
    """
    data = load_result(key)
    if data is None:
        print(f"[{key}] not baked yet.")
        print(f"  Produce it with:  {how}")
        print("  Then re-run this cell. (The pre-baked files ship with the repo;")
        print("   you only need this if you are rebuilding them yourself.)")
    return data


PREBAKED = not CAP["gpu"]   # charts get a watermark when we are replaying

## 1. A template, end to end

Read one template and convince yourself the gold cannot be wrong.

In [ ]:
from llm_utils.gen_tasks import FAMILIES, TEMPLATES, instantiate, _pools
import random, sqlite3

t = FAMILIES["revenue_by_group"]
print("family :", t.family, f"({t.level})")
print("gold   :", t.gold)
print("slots  :", t.slots, " variants:", len(t.variants), " paraphrases:", len(t.questions))

con = sqlite3.connect("data/shop.db"); pools = _pools(con); con.close()
rng = random.Random(3)
for _ in range(3):
    ex = instantiate(t, rng, pools)
    print(f"\n  Q: {ex['question']}")
    print(f"  G: {ex['gold']}")

The question and the query are formatted from the **same namespace**. A value
that appears in one necessarily appears in the other - that shared namespace is
the entire correctness argument.

Slot values are drawn **from the database**, not from constants: a price
threshold is a real price quantile, a product name is a real product. That is
what keeps every generated task answerable.

In [ ]:
print(f"{len(TEMPLATES)} templates across {len({t.family for t in TEMPLATES})} families")
for lvl in ("easy", "medium", "hard"):
    fams = [t.family for t in TEMPLATES if t.level == lvl]
    print(f"  {lvl:<7} {len(fams):>2}  {', '.join(fams[:5])}...")

## 2. Empty results are a reward-hacking surface

This one was found by reading the generated tasks by hand, and it is worth
pausing on.

`score_sql` compares **result sets**. So a gold that returns zero rows is matched
by *every* unrelated query that also returns zero rows: a typo'd literal, a
dropped join, `WHERE city='Atlantis'`.

That is not a weak training signal. It is a free reward for being wrong. The
generator rejects empty and all-NULL golds outright - including for
set-difference families like "products never ordered", where emptiness is
semantically legitimate, because **the policy cannot tell the difference between
earning an empty set and stumbling into one**.

In [ ]:
from llm_utils import fast_score_sql

gold_empty = "SELECT name FROM products WHERE category='Toys' AND price>3500;"
for wrong in ("SELECT name FROM customers WHERE city='Atlantis';",
              "SELECT name FROM products WHERE price>999999;",
              "SELECT name FROM orders WHERE status='shipped';"):
    print(f"  scores CORRECT: {fast_score_sql(wrong, gold_empty)}   <- {wrong[:58]}")
print("\n^ three unrelated queries, all 'correct' against an empty gold.")
print("  127 such candidates are rejected by the generator.")

## 3. The leakage audit

The 16 held-out tasks are the contract with repo 1. If a generated training task
duplicates one, our headline is memorisation.

Four rules, applied against **the 16 test tasks only** - repo 1's 24 *train*
tasks are training data in both repos and leak nothing. (An early version
compared against all 40 and rejected 15,205 perfectly good candidates.)

Rule 4 is the subtle one: a result-set collision only counts as leakage **in
conjunction with** question similarity. `SELECT COUNT(*) FROM orders` and a dozen
unrelated counts all return `(80,)`; returning the same number as an unrelated
test task is not leakage.

In [ ]:
import json
audit = json.load(open("data/leakage_audit.json"))
print("rejections by rule:")
for rule, n in audit["rejected_by_rule"].items():
    print(f"  {rule:<34} {n:>5}")
print(f"\nsignature-only collisions ALLOWED (not leakage): "
      f"{audit['signature_only_collisions_allowed']}")
print(f"produced: {audit['produced']}    shortfall: {audit['shortfall']}")

plt.figure(figsize=(8, 3.2))
ks = list(audit["rejected_by_rule"]); vs = [audit["rejected_by_rule"][k] for k in ks]
plt.barh(ks, vs, color="#C44E52"); plt.xlabel("candidates rejected")
plt.title("Leakage audit: what the generator threw away, and why")
plt.tight_layout(); plt.show()

### Five families are empty on purpose

Five of the 16 test tasks come from **slot-free** patterns ("products never
ordered", "orders per month"). A slot-free pattern has exactly one
instantiation - which *is* the test task - so it can contribute nothing without
leaking outright.

Those families correctly produce zero usable instances. `test_ext` covers those
five patterns through *near-variant* families instead, and we say so rather than
blurring it: for those patterns, `test_ext` measures generalization to a
**variant**, not to a fresh instance.

In [ ]:
print("slot-free test patterns (0 instances by construction):")
for f in audit["slot_free_test_families"]:
    print(f"  {f:<28} -> covered in test_ext by "
          f"{[k for k,v in audit['near_variant_of'].items() if v==f][0]}")

## 4. The four splits

In [ ]:
from llm_utils.gen_tasks import read_jsonl, split_report

splits = {n: read_jsonl(f"data/tasks_{n}_gen.jsonl")
          for n in ("train", "val", "test_ext", "train_noleak")}
for n, s in splits.items():
    r = split_report(s)
    print(f"{n:<13} n={r['n']:<5} families={r['n_families']:<3} {r['by_level']}")

print("\nsplits are mutually disjoint:")
k = lambda ts: {(t['family'], t['question']) for t in ts}
print("  train/val     ", len(k(splits['train']) & k(splits['val'])))
print("  train/test_ext", len(k(splits['train']) & k(splits['test_ext'])))
print("  val/test_ext  ", len(k(splits['val']) & k(splits['test_ext'])))

`train_noleak` is the **memorization control**: the same generator with every
test *pattern* excluded. Training on it and comparing tells us how much of any
gain is template memorisation - answered up front, rather than when someone in
the audience asks.

## 5. STaR: let the model teach itself

Sample k completions per training task at a warm temperature. Keep only the ones
`score_sql` marks correct. Fine-tune on those.

The filter *is* the method. We keep the **shortest** correct query, a mild
simplicity prior that measurably reduces rambling and incidentally makes the
model cheaper to serve.

In [ ]:
import os
from llm_utils.datasets import (star_sample, star_yield, dedup_sft, star_path,
                                read_records, write_records)
from llm_utils.local_llm import LocalLM
from llm_utils.config import base_model_4bit

train = splits["train"]
if CAP["gpu"]:
    lm = LocalLM(base_model_4bit())
    demo = star_sample(lm.as_policy(), train[:60], k=4, temperature=0.8)
    print(f"\ndemo on 60 tasks: kept {len(demo)}")
    print("yield by level:", star_yield(demo, train[:60]))
    records = read_records(star_path()) if os.path.exists(star_path()) else demo
    # PERSIST. Without this the demo pairs live only in this kernel, and
    # `bake_all.py --stage sft` dies on a missing data/star_sft.jsonl -- which
    # is exactly what the ablation cell below tells you to run.
    if not os.path.exists(star_path()):
        write_records(records, star_path())
        print(f"wrote {len(records)} pairs -> {star_path()}")
else:
    records = read_records(star_path()) if os.path.exists(star_path()) else []
    if not records:
        # Keep the return value: `baked()` IS the replay path, and discarding it
        # left `records` empty even when the artifact was present.
        records = baked("star_sft",
                  "python scripts/bake_all.py --stage star") or []

if records:
    print(f"\nfull STaR set: {len(records)} pairs, "
          f"{len(dedup_sft(records))} after dedup")
    print("yield by level:", star_yield(records, train))

### The yield curve *is* the argument for RL

Coverage runs roughly easy ~95%, medium ~60%, hard ~15%.

Read the hard bar again. On those tasks the policy almost never succeeds, so
there is **nothing to imitate** - and SFT can only imitate. No amount of extra
sampling fixes that; it is a structural limit of supervised fine-tuning.

To improve where you have no successes to copy, you must optimise expected
reward directly. That is NB3.

In [ ]:
if records:
    y = star_yield(records, train)
    plt.figure(figsize=(7, 3.6))
    plt.bar(list(y), list(y.values()),
            color=["#55A868", "#DD8452", "#C44E52"][:len(y)])
    for i, v in enumerate(y.values()):
        plt.text(i, v + 0.02, f"{v:.0%}", ha="center", fontsize=9)
    plt.ylim(0, 1.1); plt.ylabel("fraction of tasks solved at least once in k=4")
    plt.title("STaR yield by difficulty -- SFT cannot learn what was never solved")
    plt.tight_layout(); plt.show()

## 6. Train the warm start

In [ ]:
from llm_utils.datasets import to_sft_dataset
from llm_utils.trainers import (load_4bit_policy, non_finite_loss_callback,
                                t4_sft_config, vram_budget)

sft_hist = None
if CAP["gpu"] and records:
    from trl import SFTTrainer
    model, tok = load_4bit_policy()
    vram_budget("after load")
    ds = to_sft_dataset(dedup_sft(records))
    trainer = SFTTrainer(model=model, train_dataset=ds,
                         args=t4_sft_config("out/sft"),
                         callbacks=[non_finite_loss_callback()])
    trainer.train()
    vram_budget("after train")
    sft_hist = trainer.state.log_history
    save_result("nb2_sft", sft_hist)   # so a re-run replays instead of retraining
else:
    sft_hist = baked("nb2_sft",
                     "python scripts/bake_all.py --stage star,sft")

In [ ]:
from llm_utils.plotting import learning_curve
if sft_hist:
    rows = [h for h in sft_hist if "loss" in h]
    learning_curve(rows, keys=("loss",), x="step",
                   title="SFT on the STaR data", prebaked=PREBAKED)
    plt.show()

## 7. Two ablations that keep us honest

**(a) The filter is the reward.** Run the identical SFT on *unfiltered* samples -
every generation, right or wrong. If accuracy does not drop, the filter was
doing nothing.

**(b) How much is memorisation?** Train on `train_noleak` (test patterns removed
entirely) and compare on test-16 and test_ext. This is the question every
audience asks about a generated training set; we answer it before it is asked.

In [ ]:
from llm_utils import evaluate, load_result, save_result
from llm_utils.config import ADAPTER_DIR, empty_cache
from llm_utils.datasets import to_sft_dataset
from llm_utils.evaluate_batch import evaluate_jsonl, make_batch_agent
from llm_utils.gen_tasks import read_jsonl
from llm_utils.local_llm import make_local_agent
from llm_utils.trainers import non_finite_loss_callback, t4_sft_config

# Compute it if we can, replay it if we cannot, and CACHE either way -- far
# better than telling you to go run a script and come back.
ABL_TASKS = 40                       # per arm; all three share one budget or
                                     # the comparison measures budget, not method


def run_ablations(n_tasks=ABL_TASKS):
    # Three arms, one task budget. Same shape as the baked artifact.
    from trl import SFTTrainer

    noleak = read_jsonl("data/tasks_train_noleak_gen.jsonl")[:n_tasks]
    src = train[:n_tasks]
    pol = LocalLM(base_model_4bit())
    arms = {
        "filtered":   star_sample(pol.as_policy(), src, k=4, temperature=0.8),
        "unfiltered": star_sample(pol.as_policy(), src, k=4, temperature=0.8,
                                  filter_correct=False),
        "no-leak":    star_sample(pol.as_policy(), noleak, k=4, temperature=0.8),
    }
    pol.unload(); empty_cache()

    out = {"test16": {}, "test_ext": {}}
    for nm, recs in arms.items():
        if not recs:
            print(f"  {nm}: STaR kept nothing -- skipping this arm")
            continue
        m, _ = load_4bit_policy()
        ad = os.path.join(ADAPTER_DIR, f"abl-{nm}")
        t = SFTTrainer(model=m, train_dataset=to_sft_dataset(recs),
                       args=t4_sft_config(ad),
                       callbacks=[non_finite_loss_callback()])
        t.train(); t.save_model(ad)
        del m, t; empty_cache()          # or arm 3 loads on top of arm 1

        sc = LocalLM(base_model_4bit(), adapter=ad)
        r = evaluate(make_local_agent(sc), split="test")
        out["test16"][nm] = [sum(x["correct"] for x in r["records"]), r["n"]]
        rb = evaluate_jsonl(make_batch_agent(sc), "data/tasks_test_ext_gen.jsonl")
        out["test_ext"][nm] = [sum(x["correct"] for x in rb["records"]), rb["n"]]
        sc.unload(); empty_cache()
        print(f"  {nm}: {len(recs)} pairs  test16 {out['test16'][nm]}")
    return out


abl = load_result("nb2_ablations")
if abl is None and CAP["gpu"]:
    # The warm start above is still holding VRAM. Three more models are about
    # to load; drop it first or the last arm OOMs a 14.5 GB T4.
    for _n in ("trainer", "model"):
        if _n in dir():
            del globals()[_n]
    empty_cache()
    print(f"Running the ablations live on {ABL_TASKS} tasks per arm "
          f"(~10-15 min). The result caches, so this happens once.")
    try:
        abl = run_ablations()
        save_result("nb2_ablations", abl)
    except Exception as e:          # a failed ablation must not kill Run-all
        abl = None
        print(f"\nAblations did not finish: {type(e).__name__}: {e}")
        print("The rest of the notebook is unaffected. Re-run this cell to "
              "retry, or lower ABL_TASKS if it ran out of memory.")
elif abl is None:
    abl = baked("nb2_ablations",
                  "python scripts/bake_all.py --stage sft")

if abl:
    from llm_utils.plotting import bar_accuracy
    bar_accuracy({k: tuple(v) for k, v in abl["test16"].items()},
                 title="NB2 ablations on the 16 held-out tasks", prebaked=PREBAKED)
    plt.show()
    print("On test_ext (n=169, the set with real statistical power):")
    for k, v in abl.get("test_ext", {}).items():
        print("  " + report_number(tuple(v), k))

## Takeaways

1. **Construct the gold, do not guess it.** One namespace formats the question and the query, so correctness is structural rather than checked after the fact.
2. **An empty gold is a free reward for being wrong**, because `score_sql` compares result sets. Rejected outright - 127 of them.
3. Leakage is audited against the **16 test tasks only**, under four rules, and the audit is a chart rather than a claim.
4. **STaR's filter is the reward.** Unfiltered self-training just makes the model more confident in what it already does.
5. The yield curve (easy ~95%, hard ~15%) is the argument for RL: **SFT cannot learn what the policy never once got right.**

### The gap this leaves (-> NB3)

SFT imitates completions we already produced. It has no notion of "less wrong",
it cannot touch the ~85% of hard tasks it never solved, and it will happily
plateau at the ceiling of its own sampling.

To go further we have to stop imitating and start optimising expected reward
directly - which means computing an advantage, and that means GRPO.

### Exercise

1. Set `keep="first_correct"` instead of `"shortest_correct"` in `star_sample`
   and re-train. Does completion length drift up? What does that cost at serving
   time?
2. Add a template family for a query shape the schema supports but we skipped
   (window functions, `LEFT JOIN` with `IS NULL`). Run
   `python scripts/generate_tasks.py` and `pytest tests/test_generator.py` -
   the suite will tell you if your gold is wrong or your family leaks.

In [ ]:
# --- Cost / throughput meter -------------------------------------------
from llm_utils import METER, flush
print(METER)          # OpenAI spend (0 unless you ran the comparison rows)
flush()